## The Kubernetes networking model — four hard requirements

The networking spec is a small set of axioms. **Every** cluster — `kind`, EKS, a 2000-node bare-metal install — must satisfy all four:

1. **Every pod gets its own routable IP.** Not a NAT'd port map — a real IP other pods on any node can dial directly. The pod sees that IP from inside.
2. **Pods talk to all pods without NAT.** Same node or different, the source IP is preserved end to end. Reaching pod B looks identical wherever B runs.
3. **Nodes can talk to all pods (no NAT).** This is what makes kubelet probes work, and `kubectl exec` / `port-forward` proxy through.
4. **The IP a pod sees for itself is the IP others see.** No internal-vs-external address. (Docker's default networking violates this — which is why plain Docker at scale hurts.)

Together this is the **flat, IP-per-pod model**. It's easier to reason about (every connection is IP-to-IP), better for meshes and observability (sidecars sit on a real interface), and **harder to implement** — someone must make every node's pod range routable to every other's. That's the CNI's job (notebook 08).

What Kubernetes **doesn't** mandate: *how* you achieve it (overlay, BGP, cloud VPC routes, eBPF — all valid); anything about *security* (**every pod can reach every pod by default** — NetworkPolicy changes that); or bandwidth/QoS. On our map this axiom set is the foundation under the whole **Networking** box — the flat fabric that Services, Ingress, and policies all sit on top of.